# 網易雲音樂數據整合與分析 (NetEase Music Data Analysis)

此 Notebook 用於處理網易 (NetEase/NCM) 平台的結算報表。其主要功能包括：
1. **遞迴讀取** 指定資料夾內的所有網易 Excel 檔案。
2. **排除總計表**（如：总计表、汇总、Summary 等）。
3. **排除每張 Sheet 的最後一列合計資料**。
4. **提取 Sheet Name** 作為新欄位 `date`（取最後 7 位字元 YYYY-MM）。
5. **欄位自動對齊**：對齊 `Song`、`Artist`、`ISRC`、`UPC`、`Album` 欄位。
6. **動態計算點擊與營收**：
   * **Clicks**：自動加總所有包含 **「量」** 或 **「数量」** 的欄位（如播放量、下載量、使用量、銷售數量）。
   * **Revenue**：自動加總所有包含 **「费用」** 或 **「收益费用」** 的欄位。
7. **資料清洗與缺失 ISRC 填補**（若 ISRC 為空，則填入 `Song - Artist`）。
8. **產出統計報告與排行報表**。

In [42]:
# 啟用自動重新載入外部模組功能
%load_ext autoreload
%autoreload 2

import os
import glob
import pandas as pd
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
# 1. 設定輸入路徑與輸出路徑
base_path = '/Users/chu-chun/Mirror/Eva/input/sony_網易_valid/'
output_report_path = '../output/Netease_Song_Report.xlsx'
output_missing_isrc_path = '../output/Netease_Missing_ISRC_Report.xlsx'

In [44]:
# 2. 定義網易欄位對齊標準名稱的別名對照表 (不分大小寫)
netease_schema = {
    'Song': ['歌曲名', 'song', 'mv名', 'title', '歌曲名稱'],
    'Album': ['专辑', 'album', '專輯'],
    'Artist': ['艺人', 'artist', '歌手名', '歌手', '藝人'],
    'ISRC': ['isrc', '歌曲isrc', '歌曲 ISRC'],
    'UPC': ['upc', '专辑upc', '專輯 UPC']
}

# 欄位自動命名對應函數
def map_netease_columns(columns, schema):
    rename_map = {}
    for col in columns:
        col_str = str(col).strip().lower()
        for standard, aliases in schema.items():
            if col_str in [a.lower().strip() for a in aliases]:
                rename_map[col] = standard
                break
    return rename_map

# 判斷是否為總計表 (Summary Sheet) 的函數
def is_summary_sheet(sheet_name):
    ignored_keywords = ['总计', '總計', '汇总', 'summary', 'total']
    name_lower = sheet_name.lower().strip()
    return any(kw in name_lower for kw in ignored_keywords)

In [45]:
# 3. 讀取所有檔案的所有有效 Sheet，並記錄 Sheet Name 作為 date 欄位，同時動態加總點擊與收益
all_dfs = []
xlsx_files = glob.glob(os.path.join(base_path, '**/*.xlsx'), recursive=True)
# 自動排除 Excel 暫存鎖定檔案 (以 ~$ 開頭者)
xlsx_files = [f for f in xlsx_files if not os.path.basename(f).startswith('~$')]

print(f'🔍 開始掃描資料夾... 找到 {len(xlsx_files)} 個 Excel 檔案進行讀取。')

for file in xlsx_files:
    filename = os.path.basename(file)
    try: 
        xl = pd.ExcelFile(file)
        
        for sheet_name in xl.sheet_names:
            # 條件 1：不要讀總計表
            if is_summary_sheet(sheet_name):
                print(f'   [跳過] 總計表: [{sheet_name}] (檔案: {filename})')
                continue
                
            # 讀取該張 Sheet
            df_sheet = pd.read_excel(xl, sheet_name=sheet_name)
            
            # 🚨 額外條件：跳過最後一列資料 (因為是合計資料)
            if not df_sheet.empty:
                df_sheet = df_sheet.iloc[:-1]
            
            if df_sheet.empty:
                continue
                
            # 條件 2：將 sheet name 最後面 7 位字元 (例如 '2023-01') 當作新欄位 'date'
            df_sheet['date'] = sheet_name[-7:]
            
            # 計算 Clicks：動態加總所有欄位名稱含有「量」或「数量」的欄位值
            # (包括：总播放量、总下载量、销售数量、词使用量、曲使用量、原曲使用量、原伴奏使用量、消音伴奏使用量)
            click_cols = [c for c in df_sheet.columns if '量' in str(c) or '数量' in str(c)]
            if click_cols:
                df_sheet['Clicks'] = df_sheet[click_cols].fillna(0).sum(axis=1)
                print(f'      已加總點擊欄位: {click_cols}')
            else:
                df_sheet['Clicks'] = 0.0
                
            # 計算 Revenue：動態加總所有欄位名稱含有「费用」或「收益费用」的欄位值
            # (包括：本月分成收益费用、本月实际分成收益费用、本月实际销售收益费用、本月单价收益费用)
            revenue_cols = [c for c in df_sheet.columns if '费用' in str(c) or '收益费用' in str(c)]
            if revenue_cols:
                df_sheet['Revenue'] = df_sheet[revenue_cols].fillna(0).sum(axis=1)
                print(f'      已加總營收欄位: {revenue_cols}')
            else:
                df_sheet['Revenue'] = 0.0
            
            # 條件 3：欄位標準化對齊
            rename_map = map_netease_columns(df_sheet.columns, netease_schema)
            df_sheet = df_sheet.rename(columns=rename_map)
            
            # 防呆：確保核心分析欄位至少都存在 (若該 Sheet 缺漏則補為 NaN)
            for target_col in ['Song', 'Artist', 'ISRC', 'Revenue', 'Clicks']:
                if target_col not in df_sheet.columns:
                    df_sheet[target_col] = np.nan
            
            # 記錄原始檔案名稱方便追蹤
            df_sheet['source_file'] = sheet_name +'/' +filename
            
            # 保留必要的欄位合併，避免欄位過多雜亂
            keep_cols = ['Song', 'Album', 'Artist', 'ISRC', 'UPC', 'Clicks', 'Revenue', 'date', 'source_file']
            df_filtered_cols = df_sheet[[c for c in keep_cols if c in df_sheet.columns]].copy()
            
            all_dfs.append(df_filtered_cols)
            print(f'   [載入] Sheet: [{sheet_name}] (共 {len(df_filtered_cols)} 筆) 來自: {filename}')
            
    except Exception as e:
        print(f'❌ 讀取檔案失敗: {filename}, 錯誤: {e}')

# 4. 合併所有 Sheet (不同 Sheet 欄位不同會自動對齊，缺失部分填補為 NaN)
if all_dfs:
    df_raw = pd.concat(all_dfs, ignore_index=True)
    print(
        f'\n' + '='*50 + f'\n' 
        f'🎉 合併完成！共讀取 {len(xlsx_files)} 個檔案，累計 {len(df_raw)} 筆原始列資料。\n' 
        f'' + '='*50
    )
else:
    df_raw = pd.DataFrame()
    print('❌ 警告：未成功讀取到任何資料，請確認資料夾與檔案名稱。')

🔍 開始掃描資料夾... 找到 8 個 Excel 檔案進行讀取。
   [跳過] 總計表: [总计表] (檔案: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx)
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月分成收益费用']
   [載入] Sheet: [免费2023-10] (共 355 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月分成收益费用']
   [載入] Sheet: [免费2023-11] (共 356 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月分成收益费用']
   [載入] Sheet: [免费2023-12] (共 361 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月实际分成收益费用']
   [載入] Sheet: [付费2023-10] (共 359 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月实际分成收益费用']
   [載入] Sheet: [付费2023-11] (共 358 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月实际分成收益费用']
   [載入] Sheet: [付费2023-12] (共 360 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2

In [46]:
# 4. 資料清理與 ISRC 填補
df_cleaned = df_raw.copy()

if not df_cleaned.empty:
    # 4-1. 將歌名、歌手強制轉為乾淨的字串型態，並防呆小數點
    for col in ['Song', 'Artist']:
        if col in df_cleaned.columns:
            df_cleaned[col] = df_cleaned[col].apply(
                lambda x: str(int(x)) if isinstance(x, float) and x.is_integer()
                          else (str(x).strip() if pd.notna(x) else x)
            )
            
    # 4-2. 如果 ISRC 為空值，自動填入 "Song - Artist" 的組合
    if 'ISRC' in df_cleaned.columns and 'Song' in df_cleaned.columns and 'Artist' in df_cleaned.columns:
        isrc_fill = df_cleaned['Song'].fillna('UnknownSong').astype(str) + ' - ' + df_cleaned['Artist'].fillna('UnknownArtist').astype(str)
        df_cleaned['ISRC'] = df_cleaned['ISRC'].fillna(isrc_fill)
        df_cleaned.loc[df_cleaned['ISRC'].astype(str).str.strip() == '', 'ISRC'] = isrc_fill
        
    # 4-3. 統一將 ISRC 轉為大寫並去除空白
    if 'ISRC' in df_cleaned.columns:
        df_cleaned['ISRC'] = df_cleaned['ISRC'].astype(str).str.upper().str.strip()
        
    print('✅ 資料清洗與 ISRC 填補完成！')
    df_cleaned.info()
else:
    print('無資料可供清理。')

✅ 資料清洗與 ISRC 填補完成！
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11737 entries, 0 to 11736
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Song         11737 non-null  object 
 1   Album        11737 non-null  object 
 2   Artist       11737 non-null  object 
 3   ISRC         11737 non-null  object 
 4   UPC          58 non-null     float64
 5   Clicks       11737 non-null  int64  
 6   Revenue      11737 non-null  float64
 7   date         11737 non-null  object 
 8   source_file  11737 non-null  object 
dtypes: float64(2), int64(1), object(6)
memory usage: 825.4+ KB


In [47]:
# 5. 產出「缺失 ISRC 歌曲」統計報表 (一列算一筆)
if not df_cleaned.empty:
    # 判斷是否為當初缺 ISRC 的列 (欄位等於當時填補的 'Song - Artist' 格式)
    temp_fill = df_cleaned['Song'].fillna('UnknownSong').astype(str) + ' - ' + df_cleaned['Artist'].fillna('UnknownArtist').astype(str)
    isrc_missing = df_cleaned['ISRC'] == temp_fill
    df_missing = df_cleaned[isrc_missing]
    
    print(f'📊 缺失 ISRC 原始資料共計 {len(df_missing)} 行。')
    
    if not df_missing.empty:
        # 以 歌曲 + 歌手 分組，加總營收、統計筆數 (一列一筆)、列出來源檔案
        missing_report = (
            df_missing
            .groupby(['Song', 'Artist'])
            .agg(
                revenue=('Revenue', 'sum'),
                records=('Song', 'size'),  # 統計筆數 (一列算一筆)
                source_files=('source_file', lambda x: ', '.join(sorted(list(set(x.dropna())))))
            )
            .reset_index()
        )
        
        # 依照筆數由高到低進行排序
        missing_report = missing_report.sort_values(by='records', ascending=False).reset_index(drop=True)
        
        # 匯出至 Excel
        os.makedirs(os.path.dirname(output_missing_isrc_path), exist_ok=True)
        missing_report.to_excel(output_missing_isrc_path, index=False)
        
        print(f'💾 缺失 ISRC 報表已匯出至：{output_missing_isrc_path}')
        display(missing_report.head(15))
    else:
        print('🎉 太棒了！沒有任何缺失 ISRC 的歌曲。')
else:
    print('無資料可處理。')

📊 缺失 ISRC 原始資料共計 9607 行。
💾 缺失 ISRC 報表已匯出至：../output/Netease_Missing_ISRC_Report.xlsx


,Song,Artist,revenue,records,source_files
0,你的样子,罗大佑,18632.015805,179,K歌2023-01/431A202101000389-NCM-结算报表-新禧未来音乐有限公司...
1,恋曲1990,罗大佑,31149.368550,161,K歌2023-01/431A202101000389-NCM-结算报表-新禧未来音乐有限公司...
2,穿过你的黑发的我的手,罗大佑,980.780162,137,K歌2023-01/431A202101000389-NCM-结算报表-新禧未来音乐有限公司...
3,爱人同志,罗大佑,941.375040,130,K歌2023-01/431A202101000389-NCM-结算报表-新禧未来音乐有限公司...
4,光阴的故事,罗大佑,27638.736017,124,付费2023-01/431A202101000389-NCM-结算报表-新禧未来音乐有限公司...
5,东方之珠,罗大佑,6689.279952,117,K歌2023-01/431A202101000389-NCM-结算报表-新禧未来音乐有限公司...
6,爱的箴言,罗大佑,5178.156444,114,K歌2023-01/431A202101000389-NCM-结算报表-新禧未来音乐有限公司...
7,未来的主人翁,罗大佑,1774.773166,103,K歌2023-01/431A202101000389-NCM-结算报表-新禧未来音乐有限公司...
8,思念,罗大佑,612.540990,95,K歌2023-01/431A202101000389-NCM-结算报表-新禧未来音乐有限公司...
9,童年,罗大佑,53398.083785,95,K歌2023-01/431A202101000389-NCM-结算报表-新禧未来音乐有限公司...


In [ ]:
# 6. 產出「網易歌曲營收總排行報表」
if not df_cleaned.empty:
    song_report = (
        df_cleaned
        .groupby(['ISRC', 'date'])
        .agg(
            song=('Song', 'first'),
            artist=('Artist', 'first'),
            total_revenue=('Revenue', 'sum'),
            total_clicks=('Clicks', 'sum')
        )
        .reset_index()
    )
    
    # 依照總收益由高到低排序
    song_report = song_report.sort_values(by='total_revenue', ascending=False).reset_index(drop=True)
    
    # 匯出至 Excel
    os.makedirs(os.path.dirname(output_report_path), exist_ok=True)
    song_report.to_excel(output_report_path, index=False)
    
    print(f'💾 網易歌曲營收總排行報表已匯出至：{output_report_path}')
    display(song_report.head(15))
else:
    print('無資料可處理。')

💾 網易歌曲營收總排行報表已匯出至：../output/Netease_Song_Report.xlsx


,ISRC,date,total_revenue,total_clicks,records
0,风的颜色 - NINEONE#赵馨玥,2023-05,14892.743221,6779008,3
1,风的颜色 - NINEONE#赵馨玥,2023-04,14523.369629,7463550,3
2,风的颜色 - NINEONE#赵馨玥,2023-03,13206.960927,7300645,3
3,风的颜色 - NINEONE#赵馨玥,2023-01,12899.501123,6758322,3
4,风的颜色 - NINEONE#赵馨玥,2023-06,12172.248716,5519723,3
5,风的颜色 - NINEONE#赵馨玥,2023-08,12005.352779,5174878,3
6,风的颜色 - NINEONE#赵馨玥,2023-02,11816.696540,6013072,3
7,风的颜色 - NINEONE#赵馨玥,2023-07,11797.075869,5227563,3
8,风的颜色 - NINEONE#赵馨玥,2023-09,11756.644331,4718187,3
9,童年 - 罗大佑,2023-05,5429.703091,3735055,8
